# GToTR Tutorial 01: Getting Started

-----

Suppose we observe tensor responses and tensor predictors $(\mathcal{Y}_i,\mathcal{X}_i)$ independently for $i=1,2,\dots,n$.

We define the Generalized Tensor-on-Tensor Regression (GToTR) model as follows

$\displaystyle \mathcal{Y}_i\sim P(\theta_i),\quad \mu_i = E(\mathcal{Y}_i),\quad \eta_i = g(\mu_i),\quad \eta_i = \langle \mathcal{X}_i | \mathcal{B}\rangle$

where: 
  - $\mathcal{Y}_i \in \mathbb{R}^{M_1 \times ... \times M_P}$ are the responses;
  - $\mathcal{X}_i \in \mathbb{R}^{N_1 \times ... \times N_Q}$ are the covariates;
  - $\mathcal{B} \in \mathbb{R}^{N_1 \times ... \times N_Q \times M_1 \times ... \times M_P}$ is the parameter tensor of the model;
  - $\theta_i, \mu_i, \eta_i$ are all tensors of the same dimensions as $\mathcal{Y}_i$;
  - $P(\theta_i)$ is the parametric distribution of $\mathcal{Y}_i$;
  - $\mu_i$ is the expected value of the response variable;
  - $\eta_i  = g(\mu_i)$ is the link function that relates $\mu_i$ to the linear relationship; and
  - $\langle \mathcal{X}_i | \mathcal{B}\rangle$ is the linear predictor, which is a partial tensor contraction with $\langle \mathcal{X}_i | \mathcal{B}\rangle_{k_1,\dots,k_P} = \sum_{j_1,\dots,j_Q} x_{j_1,\dots,j_Q} b_{j_1,\dots,j_Q,k_1,\dots,k_P}$ .

-----

#### Estimating $\mathcal{B}$ via Maximum Likelihood Estimation

We perform parameter inference for $\mathcal{B}$ via maximum likelihood estimation. We start by modeling the parameter tensor $\mathcal{B}$ as a canonical polyadic (CP) tensor of rank $R$, i.e.,

$$\mathcal{B} =[\![{\boldsymbol \lambda}; {\boldsymbol V}_1,\dots,{\boldsymbol V}_Q,{\boldsymbol U}_1,\dots,{\boldsymbol U}_P]\!] = \sum_{r=1}^{R} \lambda_r {\boldsymbol V}_1[:,r] \circ \cdots \circ {\boldsymbol V}_Q[:,r] \circ {\boldsymbol U}_1[:,r] \circ \cdots \circ {\boldsymbol U}_P[:,r]$$

where ${\boldsymbol V}_q \in \mathbb{R}^{N_q \times R}$ for $q \in \{1, \dots, Q\}$ and ${\boldsymbol U}_p \in \mathbb{R}^{M_p \times R}$ for $p \in \{1, \dots, P\}$ are the **_factor matrices_** of $\mathcal{B}$, and ${\boldsymbol \lambda} \in \mathbb{R}^{R}$ are the **_weights_** of the rank one tensors constructed from outer products of the columns of the factor matrices.

-----

#### GToTR Software

The GToTR software approximates the maximum likelihood estimator of $\mathcal{B}$, denoted $\mathcal{\widehat B}_{MLE}$ using alternating optimization. This approach approximates each factor matrix individually while holding all other factor matrices fixed by forming a Generalized Linear Model (GLM) of the factor matrices, responses, and covariates. In the GToTR software, standard GLM solvers available from the `statsmodels` package and specialized GLM solvers aimed at improved computation performance are provided in the `gtotr` package. The GToTR parameter inference function is called `gtotr_cp` and is available in the `gtotr` package.

The steps for a user are as follows:

1. Load or create response data $\mathcal{Y}_i$'s.
2. Load or create covariate data $\mathcal{X}_i$'s.
3. Determine the distribution $P(\theta_i)$ to use in the GToTR model.
4. Determine the link function to use in the GToTR model. 
5. Choose a rank for the CP model of $\mathcal{\widehat B}_{MLE}$.
6. Compute an approximation to $\mathcal{\widehat B}_{MLE}$ using `gtotr_cp`.

-----

#### Notebook Outline

In this notebook, we illustrate how to perform parameter inference for a GToTR model for Gaussian data and the identity link function. The steps are as follows:

1. Generate $\mathcal{B}$ and $\mathcal{X}_i$'s; then generate $\mathcal{Y}_i$'s as noisy versions of $\langle \mathcal{X}_i | \mathcal{B}\rangle$.
2. Plot the distributions of $\mathcal{Y}_i$ and $\langle \mathcal{X}_i | \mathcal{B}\rangle$.
3. Approximate $\mathcal{\widehat B}_{MLE}$ using a specialized GLM solver in `gtotr`, compute $\mathcal{\widehat Y}_i = \langle \mathcal{X}_i | \mathcal{\widehat B}_{MLE}\rangle$, and plot the distribution of errors.
4. Approximate $\mathcal{\widehat B}_{MLE}$ using a standard GLM solver from `statsmodels`, compute $\mathcal{\widehat Y}_i = \langle \mathcal{X}_i | \mathcal{\widehat B}_{MLE}\rangle$, and plot the distribution of errors.
5. Demonstrate that the two approximations of $\mathcal{\widehat B}_{MLE}$ using the different approaches are nearly identical.

-----

### Import Packages

In [ ]:
from __future__ import annotations

import gtotr

gtotr.__version__

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pyttb as ttb

### Create Data

In [ ]:
# Data parameters
covariates_shape = (3, 4)
responses_shape = (4, 5, 6)
parameters_shape = covariates_shape + responses_shape
nobs = 50
rank = 2

# Gaussian noise parameter
sigma = 0.05

In [ ]:
rng = np.random.default_rng(0)
# parameter tensor:
# ktensor with user-specified rank and factor matrix elements drawn from the
# standard normal distribution (i.e., from N(0,1)) and weights large enough
# to provide a strong relationship between the covariates and responses
B = ttb.ktensor(factor_matrices=[rng.normal(size=(s, rank)) for s in parameters_shape])
# add to the weight to guarantee that there is a trend in the data
# that is stronger than the noise
B.weights += 10

# covariates:
# tensor with elements drawn from N(0,sigma**2)
Xs = ttb.tensor(rng.normal(size=(*covariates_shape, nobs), scale=sigma))

# linear predictor:
# tensor whose size equals that of the responses
XB = gtotr.utils.contract_xb_cp(B, Xs, normtype=2).to_tensor()

# responses:
# tensor with elements Y_i drawn from N([XB]_i,sigma^2)
Ys = ttb.tensor(rng.normal(XB.data, scale=sigma))

In [ ]:
# Plot responses against the linear predictor
plt.scatter(
    XB.copy().data.flatten(),
    Ys.copy().data.flatten(),
    alpha=0.5,
)
plt.xlabel("Gaussian Mean (XB)")
plt.ylabel("data (Y)")
plt.show()

### Approximate $\mathcal{\widehat B}_{MLE}$ using Specialized GToTR Solver

In [ ]:
# Use Fast GToTR specialized Gaussian/Identity solver
model = gtotr.gtotr_cp(responses=Ys, covariates=Xs, family="gaussian", link="identity")

In [ ]:
# Get the available methods for fitting this model
print(f"Fit methods: {model.fit_methods()}")

In [ ]:
%%time
# Use specialized solver for Gaussian family and Identity link
rng = np.random.default_rng(123)
results = model.fit(
    method="cp_ao_gaussian_identity", rank=rank, tolerance=1e-9, printitn=1
)
print(results.summary())

#### Estimate Responses and Plot Errors

In [ ]:
# Estimated responses using fitted model coefficient tensor
Yhat = model.contract_xb(results.coef_).to_tensor()

In [ ]:
# Plot errors
res = (Ys.data - Yhat.data).flatten()
rmse = np.sqrt(np.mean(res**2))
print(f"Gaussian/Identity Specialized Solver RMSE: {rmse:.4f}")
plt.hist(res)
plt.title("Gaussian/Identity Specialized Solver", fontsize=16)
plt.xlabel("Response Residuals (true - predicted)", fontsize=16)
plt.ylabel("Number of Observations", fontsize=16)
plt.show()

### Compute $\mathcal{\widehat B}_{MLE}$ using Standard GLM Solver

In [ ]:
%%time
rng = np.random.default_rng(123)
results_glm = model.fit(method="cp_ao_glm", rank=rank, tolerance=1e-9, printitn=1)
print(results_glm.summary())

#### Estimate Responses and Plot Errors

In [ ]:
# Estimated responses using fitted model coefficient tensor
Yhat_glm = model.contract_xb(results_glm.coef_).to_tensor()

In [ ]:
# Plot errors
res = (Ys.data - Yhat_glm.data).flatten()
rmse = np.sqrt(np.mean(res**2))
print(f"GLM Solver RMSE: {rmse:.4f}")
plt.hist(res)
plt.title("GLM Solver", fontsize=16)
plt.xlabel("Response Residuals (true - predicted)", fontsize=16)
plt.ylabel("Number of Observations", fontsize=16)
plt.show()

### Measure the Alignment of the Estimators from the Two Solvers

In [ ]:
# factor match score between estimators (perfect match is a value of 1.0)
Bhat = results.coef_
Bhat_glm = results_glm.coef_
Bhat.score(Bhat_glm)[0]

In [ ]:
# relative mean squared error between the estimators
(Bhat.full() - Bhat_glm.full()).norm() / (Bhat_glm.full()).norm()

In [ ]:
# relative mean squared error between ground truth parameters and estimator
(B.full() - Bhat.full()).norm() / (B.full()).norm()

-----

### Ideas for Further Investigation

1. How do the errors in $\mathcal{\widehat Y}_{i}$ change as a function of the Gaussian noise (`sigma`)?
2. How do the errors in $\mathcal{\widehat Y}_{i}$ change as a function of the weights used in $\mathcal{B}$?
3. What happens if you choose a rank when estimating $\mathcal{\widehat B}_{MLE}$ that is _lower_ than the rank of the true parameters $\mathcal{B}$?
4. What happens if you choose a rank when estimating $\mathcal{\widehat B}_{MLE}$ that is _higher_ than the rank of the true parameters $\mathcal{B}$?
5. How many observations of $(\mathcal{Y}_{i}, \mathcal{X}_{i})$ do you need to compute a good estimator for $\mathcal{B}$?